## Setup


In [1]:
import torch
from student_v1 import StudentEncoderBase
from loss import get_loss_functions
import torch.optim as optim
from CLIC_dataset import build_trainloader



import NeuralCompression.neuralcompression.functional as ncF
device = "cuda" if torch.cuda.is_available() else "cpu"

/home/iot/Desktop/IoT/NeuralCompression/neuralcompression/__init__.py:21: UserWarning: Could not retrieve neuralcompression version!
  warnings.warn("Could not retrieve neuralcompression version!")


In [2]:
# Import GAN Model

model = torch.hub.load("facebookresearch/NeuralCompression", "msillm_quality_vlo1")
model = model.eval()
model.update()
model.update_tensor_devices("compress")

# Freeze Model
for p in model.parameters():
    p.requires_grad = False


# Setup Teacher/Student
teacher = model.encoder
student = StudentEncoderBase()

Using cache found in /home/iot/.cache/torch/hub/facebookresearch_NeuralCompression_main
/home/iot/.cache/torch/hub/facebookresearch_NeuralCompression_main/neuralcompression/__init__.py:21: UserWarning: Could not retrieve neuralcompression version!
  warnings.warn("Could not retrieve neuralcompression version!")


In [3]:
msssim_loss, vgg_perceptual, distillation_loss = get_loss_functions()
vgg_perceptual = vgg_perceptual.to(device)

In [4]:
alpha_hint1 = 1.0   # weight for first hint loss
alpha_hint2 = 1.0   # weight for second hint loss
beta_latent = 1.0   # weight for latent loss
gamma_msssim = 0.1 # weight for MS-SSIM loss
gamma_perc = 0.01    # weight for perceptual loss

## Training

In [5]:
# Feature storage for hints
teacher_feats = {}
student_feats = {}

# Hook registration
# Assuming teacher.encoder layers accessible as .conv1, .conv2, etc.
def get_teacher_hook(name):
    def hook(module, input, output):
        teacher_feats[name] = output.detach().cpu()
    return hook

def get_student_hook(name):
    def hook(module, input, output):
        student_feats[name] = output.detach().cpu()
    return hook

# Register hooks on desired hint layers
teacher.blocks[1].register_forward_hook(get_teacher_hook('hint1'))
teacher.blocks[3].register_forward_hook(get_teacher_hook('hint2'))

student.blocks[1].register_forward_hook(get_student_hook('hint1'))
student.blocks[3].register_forward_hook(get_student_hook('hint2'))

In [6]:
optimizer = optim.Adam(student.parameters(), lr=1e-3)
#scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.9)

Freeze the Teacher

In [7]:
from tqdm import tqdm
import matplotlib.pyplot as plt

teacher.to(device)
model.decoder.to(device)
student.to(device)

def train_epoch(dataloader, epoch=None):
    student.train()
    teacher.eval()
    running_loss = 0.0
    total_hint1_loss = 0.0
    total_hint2_loss = 0.0
    total_latent_loss = 0.0
    total_ssim_loss = 0.0
    total_perc_loss = 0.0


    student.to(device)
    # Add TQDM loader
    loop = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch if epoch is not None else ''}")

    for i, x in loop:
        x = x.to(device)
        # Padding Correction        
        x, (_, _) = ncF.pad_image_to_factor(x, model._factor)
        
        
        # Latent Generation
        with torch.no_grad():
            t_latent = teacher(x)
        
        #teacher.to("cpu")
        
        s_latent = student(x)

        optimizer.zero_grad()

        # Hint losses (logit matching)
        hint1_loss = distillation_loss(student_feats['hint1'].to(device), teacher_feats['hint1'].to(device))
        hint2_loss = distillation_loss(student_feats['hint2'].to(device), teacher_feats['hint2'].to(device))
        latent_loss = distillation_loss(s_latent, t_latent)

        
        #Reconstruction via Teacher Decoder
        with torch.no_grad():
            
            x_recon = model.decoder(s_latent)
            #model.decoder.to("cpu")
            
        
        #Recon Loss
        perc_loss = vgg_perceptual(x, x_recon)
        ssim_loss = msssim_loss(x, x_recon)

        # Cumulative Loss
        loss = (alpha_hint1 * hint1_loss
                + alpha_hint2 * hint2_loss
                + beta_latent * latent_loss
                + gamma_msssim * ssim_loss
                + gamma_perc * perc_loss)

        # Backprop and optimize
        loss.backward()
        optimizer.step()

        # Accumulate loss values
        running_loss += loss.item() * x.size(0)
        total_hint1_loss += hint1_loss.item() * x.size(0)
        total_hint2_loss += hint2_loss.item() * x.size(0)
        total_latent_loss += latent_loss.item() * x.size(0)
        total_ssim_loss += ssim_loss.item() * x.size(0)
        total_perc_loss += perc_loss.item() * x.size(0)

    # Average losses
    dataset_size = len(dataloader.dataset)
    epoch_loss = running_loss / dataset_size
    avg_hint1_loss = total_hint1_loss / dataset_size
    avg_hint2_loss = total_hint2_loss / dataset_size
    avg_latent_loss = total_latent_loss / dataset_size
    avg_ssim_loss = total_ssim_loss / dataset_size
    avg_perc_loss = total_perc_loss / dataset_size

    print(f"\n[Epoch {epoch}] Component-wise Losses:")
    print(f"Hint1 Loss:     {avg_hint1_loss:.4f}")
    print(f"Hint2 Loss:     {avg_hint2_loss:.4f}")
    print(f"Latent Loss:    {avg_latent_loss:.4f}")
    print(f"MS-SSIM Loss:   {avg_ssim_loss:.4f}")
    print(f"Perceptual Loss:{avg_perc_loss:.4f}")
    print(f"Total Loss:     {epoch_loss:.4f}")

    # Plot reconstructed image after the epoch
    x_vis = x[0].detach().cpu().permute(1, 2, 0)
    x_recon_vis = x_recon[0].detach().cpu().permute(1, 2, 0)

    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].imshow(x_vis)
    axs[0].set_title("Original Image")
    axs[0].axis('off')

    axs[1].imshow(x_recon_vis)
    axs[1].set_title("Reconstructed Image")
    axs[1].axis('off')

    plt.suptitle(f"Reconstruction at Epoch {epoch}")
    plt.show()

    return epoch_loss


## Dataset

In [8]:
train_loader = build_trainloader(batch_size=32)

In [ ]:
# Example training loop
num_epochs = 10

for epoch in range(num_epochs):
    torch.cuda.empty_cache()  # Frees cached memory (not allocated memory)
    train_loss = train_epoch(train_loader)
    ##scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}")
    torch.cuda.synchronize()


Epoch :  92%|█████████▏| 144/157 [01:19<00:07,  1.64it/s]

In [ ]:
torch.save(student.state_dict(), "/home/iot/Desktop/IoT/models/student800_100epochs-1e3")

## Quantization Aware Training

In [ ]:
# Custom QAT configuration (adjust as needed)
qat_config = QConfig(
    activation=default_fake_quant.with_args(observer=torch.ao.quantization.MovingAverageMinMaxObserver,
                                           quant_min=0,
                                           quant_max=255,
                                           dtype=torch.quint8),
    weight=default_weight_fake_quant.with_args(observer=torch.ao.quantization.MinMaxObserver,
                                              quant_min=-128,
                                              quant_max=127,
                                              dtype=torch.qint8)
)

# Apply configuration to student model
student.qconfig = qat_config

In [ ]:
# Prepare model for QAT (inserts fake quantization modules)
student_prepared = prepare_qat(student, inplace=False).to(device)
student = student_prepared
# If using FX Graph Mode (recommended for complex models):
# qconfig_mapping = get_default_qat_qconfig_mapping()
# student_prepared = prepare_fx(student, qconfig_mapping, example_inputs=torch.randn(1,3,224,224).to(device)

In [ ]:
# Set to evaluation mode and convert
student_prepared.eval()
student_quantized = convert(student_prepared, inplace=False)

# For FX Graph Mode:
# student_quantized = convert_fx(student_prepared)

# Verify quantization
print(student_quantized)